## 1. Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


CUDA available: True
GPU: Tesla T4


## 2. Install dependencies

In [2]:
!pip install -q pillow tqdm opencv-python-headless scikit-image scipy pandas
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


torch 2.10.0+cu128
torchvision 0.25.0+cu128


In [3]:
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

## 3. Recreate the project files

The transformer architecture (tested component-by-component: window partition/reverse round-trip, attention shapes, shifted-window masking, and the full network at both even and odd input sizes), the training script, and the same `DegradedPairDataset` used for Method 2.


In [4]:
import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)


In [5]:
%%writefile models/__init__.py



Writing models/__init__.py


In [6]:
%%writefile models/transformer_restoration.py
"""
Method 3: pure transformer-based regression restoration -- a windowed
self-attention architecture (Swin Transformer-style, following SwinIR),
applied to damaged photo restoration the way MDTNet applies it
specifically to old photos.

Architecturally distinct from your other two methods on purpose:
  - No adversarial training (unlike Method 1's translation network)
  - No iterative generative sampling (unlike Method 2's diffusion stage)
  - Just self-attention layers trained with a plain reconstruction loss --
    a single deterministic forward pass, nothing else

This is DELIBERATELY scaled down from the full SwinIR/MDTNet configs used
in their papers (which use embed_dim=180, 6 groups of 6 blocks each --
heavy, multi-GPU-scale models). Here: embed_dim=60, 4 groups of 4 blocks,
6 attention heads. Same architectural family and mechanism, thesis-scale
compute budget. Worth stating explicitly as a scope decision, same as the
U-Net-instead-of-SwinIR choice made for DiffBIR's Stage 1.

Unlike a U-Net, this network does NOT downsample -- windowed attention
operates at the input's full resolution throughout, using shifted windows
across blocks to let information flow between windows (this is literally
Swin's whole trick: local attention within a window is cheap, and shifting
the window grid between blocks lets far-apart pixels influence each other
over several layers without ever computing full-image attention, which
would be too expensive).
"""

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
 
 
def window_partition(x: torch.Tensor, window_size: int) -> torch.Tensor:
    """(B, H, W, C) -> (num_windows*B, window_size, window_size, C)"""
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows
 
 
def window_reverse(windows: torch.Tensor, window_size: int, H: int, W: int) -> torch.Tensor:
    """Inverse of window_partition: (num_windows*B, window_size, window_size, C) -> (B, H, W, C)"""
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x
 
 
class WindowAttention(nn.Module):
    """Multi-head self-attention restricted to a local window, with a
    learnable relative position bias (standard Swin design -- lets the
    model learn "how much should a pixel attend to its neighbor 3 steps
    to the left" independent of where in the image that pair occurs)."""
 
    def __init__(self, dim: int, window_size: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
 
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)
 
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, N, N
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # N, N, 2
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)  # N, N
        self.register_buffer("relative_position_index", relative_position_index)
 
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)
 
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # x: (num_windows*B, N, C) where N = window_size * window_size
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
 
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)  # B_, num_heads, N, N
 
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(N, N, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
 
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
 
        attn = self.softmax(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x
 
 
class SwinTransformerBlock(nn.Module):
    """One transformer block: (optionally shifted) windowed attention +
    residual, then an MLP + residual. Blocks alternate between
    shift_size=0 (regular windows) and shift_size=window_size//2 (shifted
    windows) -- that alternation is what lets information cross window
    boundaries across the depth of the network."""
 
    def __init__(self, dim: int, num_heads: int, window_size: int = 8, shift_size: int = 0,
                 mlp_ratio: float = 2.0):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.shift_size = shift_size
 
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size=window_size, num_heads=num_heads)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim),
        )
 
    def _compute_attn_mask(self, H: int, W: int, device):
        """Builds the mask that prevents attention across the artificial
        boundary created by cyclically shifting the window grid -- without
        this, shifted windows would let pixels 'wrap around' the image
        edges and attend to unrelated content on the opposite side."""
        img_mask = torch.zeros((1, H, W, 1), device=device)
        h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1
 
        mask_windows = window_partition(img_mask, self.window_size)
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        return attn_mask
 
    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        # x: (B, H*W, C)
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size for given H, W"
 
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)
 
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            attn_mask = self._compute_attn_mask(H, W, x.device)
        else:
            shifted_x = x
            attn_mask = None
 
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
 
        attn_windows = self.attn(x_windows, mask=attn_mask)
 
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)
 
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
 
        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x
 
 
class RSTB(nn.Module):
    """Residual Swin Transformer Block: a stack of SwinTransformerBlocks
    (alternating regular/shifted windows) at one resolution, followed by a
    conv layer, wrapped in a residual connection around the whole group.
    This is SwinIR's mid-level building block -- several RSTBs stacked in
    sequence form the network's "deep feature extraction" stage."""
 
    def __init__(self, dim: int, depth: int, num_heads: int, window_size: int = 8,
                 use_checkpoint: bool = False):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, num_heads=num_heads, window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
            )
            for i in range(depth)
        ])
        self.conv = nn.Conv2d(dim, dim, kernel_size=3, padding=1)
 
    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        # x: (B, H*W, C)
        shortcut = x
        for block in self.blocks:
            if self.use_checkpoint and self.training:
                # Recomputes this block's activations during backward instead
                # of storing them -- trades some extra compute time for a
                # large memory reduction, since attention matrices at
                # image_size=256 are the dominant memory cost (see the
                # OutOfMemoryError this was added to fix). use_reentrant=False
                # is the current recommended checkpoint mode.
                x = checkpoint(block, x, H, W, use_reentrant=False)
            else:
                x = block(x, H, W)
        B, L, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.conv(x)
        x = x.flatten(2).transpose(1, 2)
        return shortcut + x
 
 
class TransformerRestorationNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 3, embed_dim: int = 60,
                 depths=(4, 4, 4, 4), num_heads: int = 6, window_size: int = 8,
                 use_checkpoint: bool = False):
        super().__init__()
        self.window_size = window_size
        self.embed_dim = embed_dim
 
        # Shallow feature extraction -- a single conv, same role as the
        # first conv in a U-Net, just without any downsampling
        self.conv_first = nn.Conv2d(in_channels, embed_dim, kernel_size=3, padding=1)
 
        # Deep feature extraction: a sequence of RSTB groups, all operating
        # at the SAME (full) spatial resolution -- no downsample/upsample
        # anywhere in this network, unlike the U-Net used for Method 2
        self.layers = nn.ModuleList([
            RSTB(dim=embed_dim, depth=d, num_heads=num_heads, window_size=window_size,
                 use_checkpoint=use_checkpoint)
            for d in depths
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1)
 
        # Reconstruction
        self.conv_last = nn.Conv2d(embed_dim, out_channels, kernel_size=3, padding=1)
 
    def _pad_to_window_multiple(self, x: torch.Tensor):
        """Windowed attention requires H and W to be divisible by
        window_size. Reflect-pads up to the next multiple, and returns the
        original size so the output can be cropped back down."""
        _, _, H, W = x.shape
        pad_h = (self.window_size - H % self.window_size) % self.window_size
        pad_w = (self.window_size - W % self.window_size) % self.window_size
        if pad_h or pad_w:
            x = torch.nn.functional.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        return x, H, W
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, orig_H, orig_W = self._pad_to_window_multiple(x)
 
        shallow_feat = self.conv_first(x)
 
        B, C, H, W = shallow_feat.shape
        feat = shallow_feat.flatten(2).transpose(1, 2)  # (B, H*W, C)
 
        for layer in self.layers:
            feat = layer(feat, H, W)
        feat = self.norm(feat)
 
        feat = feat.transpose(1, 2).view(B, C, H, W)
        feat = self.conv_after_body(feat)
 
        # Global residual: the network predicts a correction on top of the
        # shallow features, rather than reconstructing from nothing --
        # same principle as skip connections in the U-Net method, applied
        # once at the whole-network level instead of per-layer
        feat = feat + shallow_feat
 
        out = self.conv_last(feat)
        out = torch.tanh(out)  # match the dataset's [-1, 1] normalization
 
        return out[:, :, :orig_H, :orig_W]
 
 
def transformer_regression_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor = None,
                                 l1_weight: float = 1.0, damage_weight: float = 5.0):
    """Pure L1 regression loss -- no adversarial term, no diffusion
    sampling. This is the defining characteristic of this method: a single
    deterministic forward pass trained to minimize pixel-wise error.
 
    If `mask` is provided (the same damage mask used to composite the
    input, convention: 1.0 = clean, 0.0 = fully damaged), damaged pixels
    get amplified weight in the loss via `damage_weight`. Without this,
    damaged pixels are typically a small fraction of the total image, so
    the network can achieve a steadily-improving average loss by learning
    to reproduce the image well overall while barely learning to actually
    repair the damage -- the "loss goes down but the output looks
    unchanged" failure mode. damage_weight=5.0 means a fully-damaged pixel
    contributes 6x the loss weight of a fully-clean pixel (1.0 base +
    5.0 extra), forcing the gradient signal from those sparse regions to
    actually matter rather than being diluted by everything else.
 
    Returns (weighted_loss_for_backprop, plain_unweighted_l1_for_logging)
    -- the second value stays comparable to loss values from before this
    fix existed, so you can still judge overall reconstruction quality
    even while training optimizes the weighted version."""
    per_pixel_l1 = torch.abs(pred - target)
    plain_l1 = per_pixel_l1.mean()
 
    if mask is None:
        return l1_weight * plain_l1, plain_l1
 
    weight_map = 1.0 + damage_weight * (1.0 - mask)
    weighted_l1 = (per_pixel_l1 * weight_map).mean()
    return l1_weight * weighted_l1, plain_l1


Writing models/transformer_restoration.py


In [7]:
%%writefile data/__init__.py



Writing data/__init__.py


In [8]:

%%writefile data/degraded_pair_dataset.py
"""
Dataset for Stage 1 training: pairs of (damaged, clean) images, where the
damage comes from YOUR existing FilmDamageSimulator mask pool (generated
via generate_synthetic_only.py) rather than DiffBIR's generic synthetic
blur/noise/JPEG degradation pipeline.

Masks are composited onto clean images on the fly (mask value 255 = clean,
toward 0 = damaged), so a given clean image can pair with a different
random mask each epoch -- more effective training variety than
pre-generating a fixed set of damaged/clean pairs once.

Blend mode matches composite_damage.py's two options:
  - "screen" (default): damage LIGHTENS toward white. Physically realistic
    for scratches/abrasion, where the print's emulsion is scraped away and
    the lighter paper base shows through.
  - "multiply": damage DARKENS toward black. More appropriate for damage
    that deposits dark material (soot/smut, heavy dirt, mold staining).
Since a mixed mask (e.g. scratches + smut generated together) doesn't track
which pixel came from which damage type, this is a dataset-wide setting --
if you want type-appropriate blending for a mixed mask pool, generate and
composite scratches and smut as separate mask batches with different
--blend-mode settings instead of one mixed pool.
"""

import os
import random
 
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF
 
 
class DegradedPairDataset(Dataset):
    """
    Expects:
        clean_dir/    -- folder of clean photos (e.g. a VOC2012 subset)
        masks_dir     -- one folder path, OR a list of folder paths, each
                          containing grayscale masks from
                          generate_synthetic_only.py (mask_*.png; NOT the
                          binarised_mask_*.png variants -- those are
                          thresholded and lose the soft edges that make
                          compositing look natural).
 
                          Passing multiple folders lets you keep damage
                          types in physically separate folders (e.g. one
                          for scratches, one for smut) and combine
                          whichever subset you want per run, rather than
                          always drawing from one mixed pool.
    """
 
    def __init__(self, clean_dir: str, masks_dir, image_size: int = 256, augment: bool = True,
                 blend_mode: str = "screen", white_probability: float = 0.8):
        if blend_mode not in ("screen", "multiply", "mixed"):
            raise ValueError(f"blend_mode must be 'screen', 'multiply', or 'mixed', got '{blend_mode}'")
        if not (0.0 <= white_probability <= 1.0):
            raise ValueError(f"white_probability must be between 0 and 1, got {white_probability}")
        self.clean_dir = clean_dir
        self.masks_dirs = [masks_dir] if isinstance(masks_dir, str) else list(masks_dir)
        self.image_size = image_size
        self.augment = augment
        self.blend_mode = blend_mode
        self.white_probability = white_probability
 
        valid_ext = (".jpg", ".jpeg", ".png")
        self.clean_files = [f for f in os.listdir(clean_dir) if f.lower().endswith(valid_ext)]
 
        self.mask_files = []
        for d in self.masks_dirs:
            for f in os.listdir(d):
                if f.lower().endswith(".png") and not f.startswith("binarised_mask"):
                    self.mask_files.append(os.path.join(d, f))
 
        if len(self.clean_files) == 0:
            raise ValueError(f"No clean images found in {clean_dir}")
        if len(self.mask_files) == 0:
            raise ValueError(f"No usable masks found in {self.masks_dirs} "
                              f"(looking for mask_*.png, excluding binarised_mask_*.png)")
 
        load_size = int(image_size * 1.12)
        self.clean_resize = T.Resize(load_size)
        self.image_size_final = image_size
 
    def __len__(self):
        return len(self.clean_files)
 
    def _load_clean(self, idx):
        path = os.path.join(self.clean_dir, self.clean_files[idx])
        img = Image.open(path).convert("RGB")
        return self.clean_resize(img)
 
    def _load_random_mask(self):
        path = random.choice(self.mask_files)
        mask = Image.open(path).convert("L")  # single-channel grayscale
        return mask
 
    def _synchronized_crop_and_flip(self, clean_img, mask_img):
        """Applies the SAME random crop and flip to both the clean image
        and the mask, so the damage stays spatially aligned with the
        content it's composited onto."""
        # Resize mask to match the (already resized) clean image
        mask_img = mask_img.resize(clean_img.size, Image.BILINEAR)
 
        if self.augment:
            i, j, h, w = T.RandomCrop.get_params(clean_img, output_size=(self.image_size_final, self.image_size_final))
            clean_img = TF.crop(clean_img, i, j, h, w)
            mask_img = TF.crop(mask_img, i, j, h, w)
 
            if random.random() < 0.5:
                clean_img = TF.hflip(clean_img)
                mask_img = TF.hflip(mask_img)
            # Masks (unlike photo content) are safe to rotate freely --
            # scratches/smut don't have a "correct" orientation the way a
            # photo of a person or building does.
            if random.random() < 0.5:
                angle = random.choice([90, 180, 270])
                mask_img = TF.rotate(mask_img, angle)
        else:
            clean_img = TF.center_crop(clean_img, (self.image_size_final, self.image_size_final))
            mask_img = TF.center_crop(mask_img, (self.image_size_final, self.image_size_final))
 
        return clean_img, mask_img
 
    def __getitem__(self, idx):
        try:
            clean_img = self._load_clean(idx)
            mask_img = self._load_random_mask()
        except Exception:
            return self.__getitem__(random.randrange(len(self)))
 
        clean_img, mask_img = self._synchronized_crop_and_flip(clean_img, mask_img)
 
        clean_tensor = TF.to_tensor(clean_img)          # [0, 1], shape (3, H, W)
        mask_tensor = TF.to_tensor(mask_img)             # [0, 1], shape (1, H, W)
 
        # Composite damage onto the clean image. In "mixed" mode, each
        # sample independently rolls screen vs. multiply according to
        # white_probability -- e.g. the default 0.8 means roughly 80% of
        # composited samples get light/white damage (scratches-style) and
        # 20% get dark/black damage (smut-style), rather than one fixed
        # blend applied uniformly to the whole dataset.
        if self.blend_mode == "mixed":
            sample_blend = "screen" if random.random() < self.white_probability else "multiply"
        else:
            sample_blend = self.blend_mode
 
        if sample_blend == "screen":
            # Lightens toward white at damaged (low-mask) pixels.
            damaged_tensor = 1.0 - (1.0 - clean_tensor) * mask_tensor
        else:  # "multiply"
            # Darkens toward black at damaged (low-mask) pixels.
            damaged_tensor = clean_tensor * mask_tensor
 
        # Normalize both to [-1, 1] to match the restoration network's Tanh output
        normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        clean_tensor = normalize(clean_tensor)
        damaged_tensor = normalize(damaged_tensor)
 
        return {"damaged": damaged_tensor, "clean": clean_tensor, "mask": mask_tensor}
 
 
def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Inverse of the Normalize(mean=0.5, std=0.5) above, for saving/viewing."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)
 

Writing data/degraded_pair_dataset.py


In [9]:
%%writefile train_transformer_regression.py
"""
Train the transformer regression restoration network (Method 3): pure
windowed self-attention, trained with a plain L1 loss, no adversarial
training, no diffusion sampling.

Reuses the exact same DegradedPairDataset from Method 2's Stage 1 --
clean photos + your FilmDamageSimulator masks, composited on the fly.
Nothing about the data pipeline changes between methods; only the model
architecture and loss do.

Usage:
    python train_transformer_regression.py \
        --clean-dir ./voc_data --masks-dir ./generated_masks \
        --epochs 50 --batch-size 8 --image-size 256 \
        --out-dir ./runs/transformer_regression
"""

import argparse
import os
import time
from datetime import datetime

import torch
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils
from tqdm import tqdm

from models.transformer_restoration import TransformerRestorationNet, transformer_regression_loss
from data.degraded_pair_dataset import DegradedPairDataset, denormalize


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


class Logger:
    def __init__(self, log_path):
        os.makedirs(os.path.dirname(log_path) or ".", exist_ok=True)
        self._file = open(log_path, "a", encoding="utf-8")

    def log(self, msg):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {msg}"
        print(line, flush=True)
        self._file.write(line + "\n")
        self._file.flush()

    def close(self):
        self._file.close()


def save_comparison_grid(model, batch, out_path, device, max_images=6):
    """[damaged input | model output | clean target] -- same layout as
    Method 2's Stage 1 sample grids, for direct visual comparison between
    methods later."""
    model.eval()
    with torch.no_grad():
        n = min(max_images, batch["damaged"].shape[0])
        damaged = batch["damaged"][:n].to(device)
        clean = batch["clean"][:n].to(device)
        restored = model(damaged)
        comparison = torch.cat([denormalize(damaged), denormalize(restored), denormalize(clean)], dim=0)
        vutils.save_image(comparison, out_path, nrow=n)
    model.train()


@torch.no_grad()
def run_validation(model, val_dataloader, device, damage_weight=5.0):
    """Runs one full pass over the validation set with gradients disabled.
    Returns (weighted_loss, plain_l1) -- weighted_loss is used for best.pt
    checkpoint selection (matching the actual training objective), plain_l1
    is reported alongside for comparability with pre-fix runs and with the
    unweighted PSNR/SSIM/LPIPS metrics evaluate.py reports later."""
    model.eval()
    total_weighted, total_plain, n_batches = 0.0, 0.0, 0
    for batch in val_dataloader:
        damaged = batch["damaged"].to(device, non_blocking=True)
        clean = batch["clean"].to(device, non_blocking=True)
        mask = batch["mask"].to(device, non_blocking=True)
        restored = model(damaged)
        weighted_loss, plain_l1 = transformer_regression_loss(restored, clean, mask=mask,
                                                                damage_weight=damage_weight)
        total_weighted += weighted_loss.item()
        total_plain += plain_l1.item()
        n_batches += 1
    model.train()
    if n_batches == 0:
        return None, None
    return total_weighted / n_batches, total_plain / n_batches


def main():
    parser = argparse.ArgumentParser(description="Train the transformer regression restoration network.")
    parser.add_argument("--clean-dir", type=str, required=True)
    parser.add_argument("--masks-dir", type=str, required=True, nargs="+",
                         help="one or more mask folders; pass multiple to combine damage types kept "
                              "in separate folders, e.g. --masks-dir ./data/masks/scratches ./data/masks/smut")
    parser.add_argument("--val-clean-dir", type=str, default=None,
                         help="optional held-out validation clean-photo folder. If set, validation loss "
                              "is computed after every epoch and the best checkpoint is saved as 'best.pt'.")
    parser.add_argument("--val-masks-dir", type=str, default=None, nargs="+",
                         help="required alongside --val-clean-dir; same multi-folder format as --masks-dir")
    parser.add_argument("--blend-mode", type=str, choices=["screen", "multiply", "mixed"], default="screen",
                         help="'mixed' (default) randomly picks screen/multiply per-sample using --white-probability")
    parser.add_argument("--white-probability", type=float, default=0.8,
                         help="only used with --blend-mode mixed; fraction of samples using screen (white) blend")
    parser.add_argument("--damage-weight", type=float, default=5.0,
                         help="extra loss weight applied to damaged pixels (on top of the base weight of "
                              "1.0 everywhere) -- without this, damaged pixels are typically a small "
                              "fraction of the image, so the network can achieve a good average loss by "
                              "reproducing the image well overall while barely learning to repair damage. "
                              "Set to 0 to disable and use plain unweighted L1 (the old behavior).")
    parser.add_argument("--stop-after-epoch", type=int, default=1,
                         help="minimum epoch before early-stopping is even considered -- prevents "
                              "stopping on a lucky early epoch before training has genuinely converged. "
                              "Only used together with --stop-loss-below.")
    parser.add_argument("--stop-loss-below", type=float, default=None,
                         help="if set, training stops early once validation loss drops below this "
                              "threshold, checked at epoch >= --stop-after-epoch. Requires "
                              "--val-clean-dir to be set. Leave unset to disable.")
    parser.add_argument("--patience", type=int, default=None,
                         help="if set, training stops early if validation loss hasn't improved on "
                              "the best-so-far value for this many CONSECUTIVE validation checks -- "
                              "catches noisy plateaus/overfitting more robustly than a fixed "
                              "threshold like --stop-loss-below, since a single epoch happening to "
                              "dip below a threshold due to noise doesn't necessarily mean training "
                              "has genuinely stopped improving. Also respects --stop-after-epoch. "
                              "Leave unset to disable.")
    parser.add_argument("--out-dir", type=str, default="./runs/transformer_regression")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--embed-dim", type=int, default=60,
                         help="channel width of the transformer features -- scaled down from full "
                              "SwinIR/MDTNet's 180 for thesis-scale compute")
    parser.add_argument("--depths", type=str, default="4,4,4,4",
                         help="comma-separated block count per RSTB group, e.g. '4,4,4,4' for 4 groups "
                              "of 4 blocks each -- scaled down from SwinIR's typical 6 groups of 6")
    parser.add_argument("--num-heads", type=int, default=6)
    parser.add_argument("--window-size", type=int, default=8)
    parser.add_argument("--use-checkpoint", action="store_true",
                         help="gradient checkpointing -- trades extra compute time for substantially "
                              "lower memory use, recommended at image-size 256+ to avoid CUDA OOM")
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--sample-every", type=int, default=200)
    parser.add_argument("--steps-per-epoch", type=int, default=None)
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--val-every", type=int, default=1,
                         help="run validation every N epochs (only used if --val-clean-dir is set)")
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--amp", action="store_true")
    args = parser.parse_args()

    depths = tuple(int(d) for d in args.depths.split(","))

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    logger = Logger(os.path.join(args.out_dir, "train_log.txt"))
    log = logger.log

    device = torch.device(args.device)
    log(f"Using device: {device}")
    if device.type == "cuda":
        log(f"  GPU: {torch.cuda.get_device_name(device)}")
    cpu_count = os.cpu_count()
    log(f"  CPUs available: {cpu_count}, --num-workers set to {args.num_workers}")
    if args.num_workers > cpu_count:
        log(f"  Warning: --num-workers ({args.num_workers}) exceeds available CPUs ({cpu_count}).")

    log("Building dataset index...")
    dataset = DegradedPairDataset(args.clean_dir, args.masks_dir, image_size=args.image_size,
                                   augment=True, blend_mode=args.blend_mode, white_probability=args.white_probability)
    log(f"Loaded {len(dataset)} clean images, {len(dataset.mask_files)} damage masks "
        f"(blend_mode={args.blend_mode})")

    if args.steps_per_epoch:
        num_samples = args.steps_per_epoch * args.batch_size
        sampler = RandomSampler(dataset, replacement=True, num_samples=num_samples)
        dataloader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))
        log(f"Using --steps-per-epoch {args.steps_per_epoch}: {num_samples} images/epoch")
    else:
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))

    log("Fetching a fixed sample batch for visualization...")
    fixed_batch = next(iter(dataloader))
    log("Dataset ready.")

    val_dataloader = None
    if args.val_clean_dir:
        if not args.val_masks_dir:
            raise ValueError("--val-clean-dir requires --val-masks-dir")
        log(f"Building validation dataset from {args.val_clean_dir} / {args.val_masks_dir}...")
        val_dataset = DegradedPairDataset(args.val_clean_dir, args.val_masks_dir, image_size=args.image_size,
                                           augment=False, blend_mode=args.blend_mode, white_probability=args.white_probability)
        val_dataloader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False,
                                     num_workers=args.num_workers, drop_last=False,
                                     pin_memory=(device.type == "cuda"))
        log(f"Loaded {len(val_dataset)} validation clean images")

    best_val_loss = float("inf")
    epochs_since_improvement = 0

    model = TransformerRestorationNet(
        in_channels=3, out_channels=3, embed_dim=args.embed_dim,
        depths=depths, num_heads=args.num_heads, window_size=args.window_size,
        use_checkpoint=args.use_checkpoint,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.999))

    use_amp = args.amp and device.type == "cuda"
    if args.amp and device.type != "cuda":
        log("Note: --amp has no effect on CPU, ignoring.")
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    start_epoch = 1
    global_step = 0
    if args.resume:
        log(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in model.parameters())
    log(f"Model has {num_params:,} parameters (embed_dim={args.embed_dim}, depths={depths}, "
        f"num_heads={args.num_heads}, window_size={args.window_size})")
    log(f"Starting training: epochs {start_epoch}-{args.epochs}")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running_loss = 0.0
        running_data_time, running_compute_time = 0.0, 0.0

        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}/{args.epochs}", unit="batch", leave=False)
        batch_end_time = time.time()

        for batch in progress_bar:
            data_time = time.time() - batch_end_time
            compute_start = time.time()

            damaged = batch["damaged"].to(device, non_blocking=True)
            clean = batch["clean"].to(device, non_blocking=True)
            mask = batch["mask"].to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.amp.autocast(device.type, enabled=use_amp):
                restored = model(damaged)
                loss, l1 = transformer_regression_loss(restored, clean, mask=mask,
                                                         damage_weight=args.damage_weight)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if device.type == "cuda":
                torch.cuda.synchronize()
            compute_time = time.time() - compute_start

            running_loss += loss.item()
            running_data_time += data_time
            running_compute_time += compute_time
            global_step += 1

            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

            if args.log_every and global_step % args.log_every == 0:
                log(f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s")

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_comparison_grid(model, fixed_batch, sample_path, device)
                log(f"  Saved sample grid: {sample_path}")

            batch_end_time = time.time()

        n_batches = len(dataloader)
        elapsed = time.time() - start_time
        log(f"[Epoch {epoch}/{args.epochs}] loss={running_loss / n_batches:.4f} "
            f"avg_data_time={running_data_time / n_batches:.3f}s "
            f"avg_compute_time={running_compute_time / n_batches:.3f}s "
            f"epoch_time={format_duration(time.time() - epoch_start)} "
            f"total_elapsed={format_duration(elapsed)}")

        if val_dataloader is not None and epoch % args.val_every == 0:
            val_loss, val_plain_l1 = run_validation(model, val_dataloader, device, damage_weight=args.damage_weight)
            log(f"  [Validation] epoch {epoch}: weighted_loss={val_loss:.4f} plain_l1={val_plain_l1:.4f}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_since_improvement = 0
                best_path = os.path.join(checkpoints_dir, "best.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "val_loss": val_loss,
                    "val_plain_l1": val_plain_l1,
                    "args": vars(args),
                }, best_path)
                log(f"  New best validation loss ({val_loss:.4f}) -- saved {best_path}")
            else:
                epochs_since_improvement += 1

            if (args.stop_loss_below is not None and epoch >= args.stop_after_epoch
                    and val_loss < args.stop_loss_below):
                log(f"  Validation loss ({val_loss:.4f}) dropped below --stop-loss-below "
                    f"({args.stop_loss_below}) at epoch {epoch} (>= --stop-after-epoch "
                    f"{args.stop_after_epoch}) -- stopping early.")
                ckpt_path = os.path.join(checkpoints_dir, f"transformer_regression_epoch{epoch:04d}.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "args": vars(args),
                }, ckpt_path)
                log(f"  Saved final checkpoint before stopping: {ckpt_path}")
                break

            if (args.patience is not None and epoch >= args.stop_after_epoch
                    and epochs_since_improvement >= args.patience):
                log(f"  Validation loss hasn't improved on the best value ({best_val_loss:.4f}) for "
                    f"{epochs_since_improvement} consecutive checks (>= --patience {args.patience}) "
                    f"at epoch {epoch} -- stopping early.")
                ckpt_path = os.path.join(checkpoints_dir, f"transformer_regression_epoch{epoch:04d}.pt")
                torch.save({
                    "epoch": epoch, "global_step": global_step,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "args": vars(args),
                }, ckpt_path)
                log(f"  Saved final checkpoint before stopping: {ckpt_path}")
                log(f"  Note: best.pt (val_loss={best_val_loss:.4f}) is likely more useful than this "
                    f"final checkpoint for downstream use, given training had stopped improving.")
                break

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"transformer_regression_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch, "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            log(f"  Saved checkpoint: {ckpt_path}")

    log(f"Training complete. Total time: {format_duration(time.time() - start_time)}")
    logger.close()


if __name__ == "__main__":
    main()

Writing train_transformer_regression.py


In [10]:
%%writefile composite_damage.py
"""
Composite a generated damage mask (from generate_synthetic_only.py or
damage_generator.py) onto a clean target image, producing a damaged/clean
training pair for restoration model training.

The mask convention from this codebase: 255 = clean/undamaged, values toward
0 = damaged (dust, dirt, scratches etc).

Two blend modes are supported:
  - "screen" (default): LIGHTENS toward white at damaged pixels. This is the
    physically realistic choice for most scratch/abrasion damage, where the
    print's emulsion is scraped away and the lighter paper base shows
    through -- old photo scratches are usually bright/white marks, not dark
    ones.
  - "multiply": DARKENS toward black at damaged pixels. More appropriate for
    damage types that genuinely deposit dark material (soot/smut, heavy
    dirt, mold staining) rather than abrading the surface.

Since a single generated mask can currently mix multiple damage types
(e.g. scratches + smut) without tracking which pixel came from which type,
this is a per-composite choice rather than automatic per-pixel selection.
If your mask pool separates damage types into different files (e.g. by
generating scratches and smut as separate mask batches), you can composite
each with the blend mode that suits it and merge afterward, rather than
using one blend mode for a mixed mask.

Usage:
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend multiply
"""

import argparse
import cv2 as cv
import numpy as np


def composite(clean_img, mask_img, blend="screen"):
    if clean_img.shape[:2] != mask_img.shape[:2]:
        mask_img = cv.resize(mask_img, (clean_img.shape[1], clean_img.shape[0]), interpolation=cv.INTER_LINEAR)

    mask_norm = mask_img.astype(np.float32) / 255.0
    if clean_img.ndim == 3 and mask_norm.ndim == 2:
        mask_norm = mask_norm[:, :, None]

    clean_f = clean_img.astype(np.float32)

    if blend == "screen":
        # Lightens toward white at damaged (low-mask) pixels.
        damaged = 255.0 - (255.0 - clean_f) * mask_norm
    elif blend == "multiply":
        # Darkens toward black at damaged (low-mask) pixels.
        damaged = clean_f * mask_norm
    else:
        raise ValueError(f"Unknown blend mode '{blend}', expected 'screen' or 'multiply'")

    return np.clip(damaged, 0, 255).astype(np.uint8)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Composite a damage mask onto a clean image.')
    parser.add_argument('--clean', required=True, help='path to the clean input image')
    parser.add_argument('--mask', required=True, help='path to the generated grayscale damage mask')
    parser.add_argument('--out', required=True, help='path to write the damaged output image')
    parser.add_argument('--blend', choices=['screen', 'multiply'], default='screen',
                         help="'screen' (default) produces light/white damage marks; "
                              "'multiply' produces dark damage marks")
    args = parser.parse_args()

    clean_img = cv.imread(args.clean, cv.IMREAD_UNCHANGED)
    mask_img = cv.imread(args.mask, cv.IMREAD_GRAYSCALE)

    damaged = composite(clean_img, mask_img, blend=args.blend)
    cv.imwrite(args.out, damaged)
    print(f"Wrote damaged image to {args.out} (blend={args.blend})")


Writing composite_damage.py


## 4. Get VOC2012 clean images — automatic, no manual download

In [11]:
import torchvision.datasets as tvds

VOC_ROOT = '/kaggle/working/voc_data'
VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')

if os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0:
    print(f'VOC2012 already present at {VOC_JPEG_DIR}, skipping download.')
else:
    os.makedirs(VOC_ROOT, exist_ok=True)
    _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)

num_images = len([f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')])
print(f'VOC2012 ready: {num_images} images at {VOC_JPEG_DIR}')


100%|██████████| 2.00G/2.00G [01:00<00:00, 33.3MB/s]


VOC2012 ready: 17125 images at /kaggle/working/voc_data/VOCdevkit/VOC2012/JPEGImages


In [12]:
# If you don't already have this in your notebook, build it once:
import random, shutil

VAL_SUBSET_DIR = '/kaggle/working/voc_val_subset/images'
os.makedirs(VAL_SUBSET_DIR, exist_ok=True)
existing_val = [f for f in os.listdir(VAL_SUBSET_DIR) if f.lower().endswith('.jpg')]
TARGET_N_VAL = 150

if len(existing_val) >= TARGET_N_VAL:
    print(f'{len(existing_val)} validation images already present, skipping.')
else:
    all_voc_images = [f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')]
    random.seed(42)
    val_subset = random.sample(all_voc_images, TARGET_N_VAL)
    for fname in val_subset:
        shutil.move(os.path.join(VOC_JPEG_DIR, fname), os.path.join(VAL_SUBSET_DIR, fname))
    print(f'Copied {len(val_subset)} validation images')

Copied 150 validation images


## 5. Generate damage masks

Clones FilmDamageSimulator and recreates `generate_synthetic_only.py`.

In [13]:
# if os.path.isdir('FilmDamageSimulator'):
#     print('FilmDamageSimulator already cloned, skipping.')
# else:
#     !git clone --depth 1 https://github.com/daniela997/FilmDamageSimulator.git


In [14]:
# %%writefile FilmDamageSimulator/damage_generator/generate_synthetic_only.py
# """
# Generate damage overlay masks using ONLY the pre-classified synthetic damage
# patches in /synthetic/<type>/ (e.g. scratches, smut), without ever touching
# the real scanned film frames in /scans/.

# This bypasses damage_generator.py's default behaviour, which always loads
# /scans/ and mixes real scanned artifact crops into the sampling pool even
# when --synthetic is passed. Here, only the folder(s) you name are loaded,
# and artifact count/size statistics are fit on those patches' own area
# distribution instead of the real-scan-derived Gamma distributions.

# Usage:
#     python generate_synthetic_only.py --types scratches,smut --height 1024 --width 1024
#     python generate_synthetic_only.py --types scratches --procedural-scratches
# """

# import os
# import argparse
# import uuid
# import random
# import numpy as np
# import pandas as pd
# import cv2 as cv
# import scipy.stats as stats
# import skimage.transform as skimage_tf

# from scans import load_images
# from generate_masks import generate_perlin_noise_2d, increase_contrast, random_perlin_with_numpy, line_scratch


# def sample_size_from_own_distribution(df, num_artifact):
#     """Fit a Gamma distribution to this dataframe's OWN artifact areas
#     (instead of a real-scan-derived one) and sample target sizes from it."""
#     areas = df['Contour Area']
#     gamma_param = stats.gamma.fit(areas, floc=0)
#     shape, _, scale = gamma_param
#     return np.random.gamma(shape, scale, num_artifact)


# def sample_closest_in_area(df, target_areas):
#     df = df.sample(frac=1).reset_index(drop=True)
#     areas = df['Contour Area']
#     indexes = []
#     for target in target_areas:
#         candidates = df.iloc[(areas - target).abs().argsort()[:15]].index.tolist()
#         index = random.choice(candidates)
#         indexes.append(index)
#         areas = areas.drop(areas.index[[index]])
#     picked = df.iloc[indexes].copy()
#     picked['Target size'] = target_areas
#     return picked


# def build_mask(target_size, per_type_dfs, per_type_counts, rescale=True, verbose=False):
#     rescale_factor = (target_size[0] / 2560 if target_size[0] <= target_size[1]
#                        else target_size[1] / 2560) if rescale else 1.

#     selected_frames = []
#     for artifact_type, df in per_type_dfs.items():
#         lo, hi = per_type_counts[artifact_type]
#         num = int(np.random.randint(lo, hi + 1))
#         if num == 0 or len(df) == 0:
#             continue
#         target_areas = sample_size_from_own_distribution(df, num)
#         picked = sample_closest_in_area(df, target_areas)
#         selected_frames.append(picked)
#         if verbose:
#             print(f"Selected {num} '{artifact_type}' artifacts")

#     if not selected_frames:
#         raise ValueError("No artifacts selected - check your --types and --min-count/--max-count")

#     selected_artifacts_df = pd.concat(selected_frames, ignore_index=True)
#     artifacts_num = len(selected_artifacts_df)

#     mask_final = np.zeros(target_size).astype(np.uint8)
#     perlin_noise = generate_perlin_noise_2d(target_size, (2, 2))
#     normalised_noise = (perlin_noise - np.min(perlin_noise)) / np.ptp(perlin_noise)
#     xs, ys = random_perlin_with_numpy(artifacts_num, normalised_noise)
#     random_angles = np.random.randint(0, 360, size=artifacts_num)

#     i = 0
#     for _, artifact_row in selected_artifacts_df.iterrows():
#         try:
#             artifact = artifact_row['Artifact'].astype(np.uint8)
#             random_scale = artifact_row['Target size'] / artifact_row['Contour Area']
#             random_angle = random_angles[i]
#             new_rescale_factor = rescale_factor * np.sqrt(random_scale)
#             artifact = skimage_tf.rescale(artifact, round(new_rescale_factor, 2), anti_aliasing=True, preserve_range=True)
#             artifact = skimage_tf.rotate(artifact, angle=random_angle, resize=True, preserve_range=True)
#             artifact_w, artifact_h = artifact.shape[:2]

#             x1 = xs[i] - artifact_w // 2
#             x2 = x1 + artifact_w
#             if x1 < 0:
#                 artifact = artifact[-x1:, :]; x1 = 0
#             if x2 > target_size[0]:
#                 artifact = artifact[:-(x2 - target_size[0]), :]; x2 = target_size[0]

#             y1 = ys[i] - artifact_h // 2
#             y2 = y1 + artifact_h
#             if y1 < 0:
#                 artifact = artifact[:, -y1:]; y1 = 0
#             if y2 > target_size[1]:
#                 artifact = artifact[:, :-(y2 - target_size[1])]; y2 = target_size[1]

#             mask_final[x1:x2, y1:y2] = np.where(
#                 artifact > mask_final[x1:x2, y1:y2], artifact, mask_final[x1:x2, y1:y2]
#             )
#             i += 1
#         except Exception:
#             i += 1
#             continue

#     mask_final = np.invert(mask_final.astype(np.uint8))
#     binarised = ((mask_final > 240) * 255).astype(np.uint8)
#     return mask_final.astype(np.uint8), binarised


# def add_procedural_scratches(mask, height, width, verbose=False):
#     """Blend in fully procedural (Perlin-noise-based) scratch lines.
#     These require NO source images at all -- real or synthetic -- so they
#     are always 'safe' to include without pulling in any scan data."""
#     num_extra_scratch = int(np.random.gamma(6, 2, 1)[0])
#     for _ in range(num_extra_scratch):
#         length = np.random.randint(10, high=max(height, width), dtype=int)
#         try:
#             scratch = line_scratch(np.array(length))
#             sw, sh = scratch.shape[:2]
#             if sw >= width or sh >= height:
#                 continue
#             x1 = np.random.randint(0, width - sw)
#             y1 = np.random.randint(0, height - sh)
#             region = mask[x1:x1 + sw, y1:y1 + sh]
#             mask[x1:x1 + sw, y1:y1 + sh] = np.minimum(region, np.invert(scratch.astype(np.uint8)))
#         except Exception:
#             continue
#     if verbose:
#         print(f"Added {num_extra_scratch} procedural scratch lines")
#     return mask


# if __name__ == '__main__':
#     parser = argparse.ArgumentParser(
#         description='Generate damage masks from ONLY classified synthetic patches (no scanned frames).'
#     )
#     parser.add_argument('--types', type=str, default='scratches,smut',
#                          help='comma-separated subfolder names under /synthetic/, '
#                               'e.g. scratches,smut,dirt,dots,hair,hair-short,lint,sprinkles,spots,stain')
#     parser.add_argument('--height', type=int, default=1024)
#     parser.add_argument('--width', type=int, default=1024)
#     parser.add_argument('--min-count', type=int, default=3, help='min number of artifacts per type')
#     parser.add_argument('--max-count', type=int, default=15, help='max number of artifacts per type')
#     parser.add_argument('--procedural-scratches', action='store_true',
#                          help='also blend in fully procedural line scratches (no source image needed)')
#     parser.add_argument('--n', type=int, default=1, help='how many masks to generate')
#     parser.add_argument('--verbose', action='store_true')
#     args = parser.parse_args()

#     abs_path = os.path.abspath(os.path.dirname(__file__))
#     synthetic_path = os.path.dirname(os.path.normpath(abs_path)) + '/synthetic/'
#     out_dir = os.path.dirname(os.path.normpath(abs_path)) + '/generated/'
#     os.makedirs(out_dir, exist_ok=True)

#     types = [t.strip() for t in args.types.split(',') if t.strip()]

#     per_type_dfs = {}
#     for t in types:
#         df = load_images(synthetic_path, t, verbose=args.verbose)
#         df['Contour Area'] = df['Non-zero pixel area']
#         per_type_dfs[t] = df
#         print(f"Loaded {len(df)} '{t}' artifact patches from /synthetic/{t}/")

#     per_type_counts = {t: (args.min_count, args.max_count) for t in types}

#     for n in range(args.n):
#         mask, binary_mask = build_mask(
#             (args.height, args.width), per_type_dfs, per_type_counts, verbose=args.verbose
#         )

#         if args.procedural_scratches:
#             mask = add_procedural_scratches(mask, args.height, args.width, verbose=args.verbose)
#             binary_mask = ((mask > 240) * 255).astype(np.uint8)

#         uid = str(uuid.uuid4())[:8]
#         tag = "_".join(types)
#         cv.imwrite(out_dir + f'mask_{tag}_{uid}.png', mask)
#         cv.imwrite(out_dir + f'binarised_mask_{tag}_{uid}.png', binary_mask)
#         print(f"[{n+1}/{args.n}] Saved mask_{tag}_{uid}.png")

#     print(f"Done. Masks written to {out_dir}")


In [15]:
# MASKS_DIR = '/kaggle/working/generated_masks'
# os.makedirs(MASKS_DIR, exist_ok=True)

# TARGET_N_MASKS = 3000
# existing_masks = [f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')]

# if len(existing_masks) >= TARGET_N_MASKS:
#     print(f'{len(existing_masks)} masks already present, skipping generation.')
# else:
#     os.chdir('/kaggle/working/FilmDamageSimulator/damage_generator')
#     import shutil
#     !python generate_synthetic_only.py --types smut \
#         --height 128 --width 128 --min-count 3 --max-count 15 --n {TARGET_N_MASKS} --verbose
#     os.chdir('/kaggle/working')
#     src_dir = 'FilmDamageSimulator/generated'
#     for fname in os.listdir(src_dir):
#         shutil.copy(os.path.join(src_dir, fname), os.path.join(MASKS_DIR, fname))

# n_masks = len([f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')])
# print(f'{n_masks} usable masks ready at {MASKS_DIR}')


In [16]:
#!ls -la /kaggle/input/datasets/dorast/generated-masks/


In [17]:
import os

MASKS_DIR = '/kaggle/input/datasets/dorast/generated-masks'
DAMAGE_TYPES = ['dirt', 'scratches', 'smut', 'spots']

MASKS_DIRS = [os.path.join(MASKS_DIR, t) for t in DAMAGE_TYPES]
MASKS_DIRS_ARG = ' '.join(MASKS_DIRS)

for d in MASKS_DIRS:
    n = len([f for f in os.listdir(d) if f.startswith('mask_')]) if os.path.isdir(d) else 0
    print(f'{d}: {n} masks')

/kaggle/input/datasets/dorast/generated-masks/dirt: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/scratches: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/smut: 3000 masks
/kaggle/input/datasets/dorast/generated-masks/spots: 3000 masks


## 6. Test: apply damage masks to a few photos

Visual sanity check — damage should render light/white (screen blend, the default).

In [18]:
# import matplotlib.pyplot as plt
# import cv2 as cv
# import random
# from composite_damage import composite

# clean_files = [f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')]
# sample_clean_files = random.sample(clean_files, 4)

# mask_files = []
# for d in MASKS_DIRS:
#     mask_files += [os.path.join(d, f) for f in os.listdir(d) if f.startswith('mask_')]
# sample_mask_files = random.sample(mask_files, 4)

# fig, axes = plt.subplots(4, 2, figsize=(6, 12))
# for i, (clean_fname, mask_path) in enumerate(zip(sample_clean_files, sample_mask_files)):
#     clean_img = cv.imread(os.path.join(VOC_JPEG_DIR, clean_fname))
#     mask_img = cv.imread(mask_path, cv.IMREAD_GRAYSCALE)
#     damaged_img = composite(clean_img, mask_img)
#     axes[i, 0].imshow(cv.cvtColor(clean_img, cv.COLOR_BGR2RGB)); axes[i,0].set_title('Clean'); axes[i,0].axis('off')
#     axes[i, 1].imshow(cv.cvtColor(damaged_img, cv.COLOR_BGR2RGB)); axes[i,1].set_title('Damaged'); axes[i,1].axis('off')
# plt.tight_layout()
# plt.show()

## 7. Quick smoke test (small config, low resolution)

Small `--embed-dim`/`--depths`/`--image-size` first, purely to confirm the pipeline runs end to end on this GPU. Not representative of final quality -- see section 9 for the real-scale config.

In [19]:

# !python train_transformer_regression.py \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIRS" \
#     --epochs 10 --batch-size 16 --image-size 128 --num-workers 2 \
#     --embed-dim 48 --depths "4,4,4,4" --num-heads 4 --window-size 8 \
#     --steps-per-epoch 400\
#     --log-every 50 --sample-every 200 --save-every 1 \
#     --amp --out-dir ./runs/baseline_check --device cuda

In [20]:
# import subprocess

# result = subprocess.run([
#     'python', 'train_transformer_regression.py',
#     '--clean-dir', VOC_JPEG_DIR, '--masks-dir', *MASKS_DIRS,
#     '--epochs', '5', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
#     '--embed-dim', '48', '--depths', '4,4,4,4', '--num-heads', '4', '--window-size', '8',
#     '--steps-per-epoch', '400',
#     '--log-every', '50', '--sample-every', '200', '--save-every', '1',
#     '--stop-loss-below', '0.05',
#     '--amp', '--out-dir', './runs/baseline_check', '--device', 'cuda',
# ])
# result.check_returncode()

In [21]:
import subprocess

result = subprocess.run([
    'python', 'train_transformer_regression.py',
    '--clean-dir', VOC_JPEG_DIR, '--val-clean-dir', VAL_SUBSET_DIR,
    '--masks-dir', *MASKS_DIRS, '--val-masks-dir', *MASKS_DIRS,
    '--epochs', '70', '--batch-size', '16', '--image-size', '128', '--num-workers', '2',
    '--embed-dim', '48', '--depths', '4,4,4,4', '--num-heads', '4', '--window-size', '8',
    '--steps-per-epoch', '400',
    '--log-every', '50', '--sample-every', '200', '--save-every', '1', '--val-every', '1',
    '--stop-loss-below', '0.01', '--patience', '6', '--stop-after-epoch', '45',
    '--amp', '--out-dir', './runs/baseline_check', '--device', 'cuda',
    '--resume', '/kaggle/input/notebooks/dorast/transformer-regression-kaggle/runs/baseline_check/checkpoints/best.pt',
])
result.check_returncode()


[2026-09-12 09:54:33] Using device: cuda
[2026-09-12 09:54:34]   GPU: Tesla T4
[2026-09-12 09:54:34]   CPUs available: 4, --num-workers set to 2
[2026-09-12 09:54:34] Building dataset index...
[2026-09-12 09:54:34] Loaded 16975 clean images, 12000 damage masks (blend_mode=screen)
[2026-09-12 09:54:34] Using --steps-per-epoch 400: 6400 images/epoch
[2026-09-12 09:54:34] Fetching a fixed sample batch for visualization...
[2026-09-12 09:54:34] Dataset ready.
[2026-09-12 09:54:34] Building validation dataset from /kaggle/working/voc_val_subset/images / ['/kaggle/input/datasets/dorast/generated-masks/dirt', '/kaggle/input/datasets/dorast/generated-masks/scratches', '/kaggle/input/datasets/dorast/generated-masks/smut', '/kaggle/input/datasets/dorast/generated-masks/spots']...
[2026-09-12 09:54:34] Loaded 150 validation clean images
[2026-09-12 09:54:34] Resuming from /kaggle/input/notebooks/dorast/transformer-regression-kaggle/runs/baseline_check/checkpoints/best.pt
[2026-09-12 09:54:35] Mod

Epoch 41/70:  12%|█▎        | 50/400 [00:50<05:43,  1.02batch/s, loss=0.0433]

[2026-09-12 09:55:25]   step 16050: data_time=0.000s compute_time=0.981s


Epoch 41/70:  25%|██▌       | 100/400 [01:40<05:07,  1.02s/batch, loss=0.0487]

[2026-09-12 09:56:15]   step 16100: data_time=0.001s compute_time=1.028s


...

Epoch 47/70:  88%|████████▊ | 350/400 [05:56<00:50,  1.02s/batch, loss=0.0473]

[2026-09-12 10:41:45]   step 18750: data_time=0.000s compute_time=1.017s


Epoch 47/70: 100%|█████████▉| 399/400 [06:46<00:01,  1.01s/batch, loss=0.0474]

[2026-09-12 10:42:36]   step 18800: data_time=0.000s compute_time=1.015s
[2026-09-12 10:42:36]   Saved sample grid: ./runs/baseline_check/samples/step_0018800.png
[2026-09-12 10:42:36] [Epoch 47/70] loss=0.0445 avg_data_time=0.001s avg_compute_time=1.015s epoch_time=6m 47s total_elapsed=48m 1s


[2026-09-12 10:42:42]   [Validation] epoch 47: weighted_loss=0.0440 plain_l1=0.0220
[2026-09-12 10:42:42]   Validation loss hasn't improved on the best value (0.0413) for 6 consecutive checks (>= --patience 6) at epoch 47 -- stopping early.
[2026-09-12 10:42:42]   Saved final checkpoint before stopping: ./runs/baseline_check/checkpoints/transformer_regression_epoch0047.pt
[2026-09-12 10:42:42]   Note: best.pt (val_loss=0.0413) is likely more useful than this final checkpoint for downstream use, given training had stopped improving.
[2026-09-12 10:42:42] Training complete. Total time: 48m 6s


## 8. View a reconstruction sample

Top row = damaged input, middle row = model output, bottom row = clean target.

In [22]:
# import glob
# from PIL import Image

# sample_files = sorted(glob.glob('runs/baseline_check/samples/*.png'))
# if sample_files:
#     img = Image.open(sample_files[-1])
#     plt.figure(figsize=(14, 7))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.title(f'Latest sample: {sample_files[-1]}')
#     plt.show()
# else:
#     print('No samples found yet — check the training cell above ran successfully.')


## 9. Baseline sanity check: short run at full-scale config

The REAL architecture size (`embed-dim 60`, `depths 4,4,4,4`, 256px) for a short run, before committing to a long unattended full run. Gives you an actual epoch-time estimate on GPU.

In [23]:
#!python train_transformer_regression.py \
 #   --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
  #  --epochs 8 --batch-size 8 --image-size 256 --num-workers 2 \
   # --embed-dim 60 --depths "4,4,4,4" --num-heads 6 --window-size 8 \
   # --log-every 20 --sample-every 50 --save-every 4 \
    #--amp --out-dir ./runs/baseline_check --device cuda


In [24]:
###version 2
#!python train_transformer_regression.py \
 #   --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
  #  --epochs 8 --batch-size 4 --image-size 256 --num-workers 2 \
   # --embed-dim 60 --depths "4,4,4,4" --num-heads 6 --window-size 8 \
   # --use-checkpoint \
   # --log-every 20 --sample-every 50 --save-every 4 \
   # --amp --out-dir ./runs/baseline_check --device cuda

In [25]:
#!python train_transformer_regression.py \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 100 --batch-size 8 --image-size 256 \
#     --embed-dim 60 --depths "4,4,4,4" --num-heads 6 --window-size 8 \
#     --amp --out-dir ./runs/transformer_regression --device cuda \
#     --resume /kaggle/working/runs/transformer_regression/checkpoints/<latest>.pt

## 10. Full training run

Once the baseline check looks right, scale up epochs. Remember to **Save Version** before a Kaggle session ends. Check `train_log.txt` inside your run's output folder for a full timestamped history.


In [26]:
# Example full run:
# !python train_transformer_regression.py \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 100 --batch-size 8 --image-size 256 --num-workers 2 \
#     --embed-dim 60 --depths "4,4,4,4" --num-heads 6 --window-size 8 \
#     --amp --out-dir ./runs/transformer_regression --device cuda

# To resume from a previous session's checkpoint:
# !python train_transformer_regression.py \
#     --clean-dir "$VOC_JPEG_DIR" --masks-dir "$MASKS_DIR" \
#     --epochs 100 --batch-size 8 --image-size 256 \
#     --embed-dim 60 --depths "4,4,4,4" --num-heads 6 --window-size 8 \
#     --amp --out-dir ./runs/transformer_regression --device cuda \
#     --resume /kaggle/working/runs/transformer_regression/checkpoints/<latest>.pt
